In [20]:
import os
#os.environ["CUDA_VISIBLE_DEVICES"] = "0"
!nvidia-smi
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import  sklearn
from sklearn.model_selection import train_test_split
import ray
import os
import math
import matplotlib.pyplot as plt
import logging
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

Wed Nov 15 05:47:19 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 520.61.05    Driver Version: 520.61.05    CUDA Version: 11.8     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA GeForce ...  On   | 00000000:05:00.0 Off |                  N/A |
|  0%   47C    P8     7W / 198W |      1MiB /  8192MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
|   1  NVIDIA GeForce ...  On   | 00000000:06:00.0 Off |                  N/A |
| 35%   

In [21]:
os.cpu_count()
ray.shutdown()

In [22]:

ray.init(num_cpus=4, num_gpus=4, logging_level=logging.WARNING)
#ray.shutdown()

Python version:,3.8.18
Ray version:,2.7.1


In [23]:
tf.get_logger().setLevel(logging.ERROR)


# Data processing

For the homework, you should download the dataset with the provided link. The provided dataset has already partitioned the data by clients.

In [24]:
NUM_CLIENT = 100
train_data = 'train_data.npy'
train_data_arr = np.load(train_data, allow_pickle=True)
test_data = 'test_data.npy'
test_data_arr = np.load(test_data, allow_pickle=True)

# Define model

In [25]:
'''
def get_model():
    model = tf.keras.models.Sequential(
        [ tf.keras.layers.Flatten(input_shape=(28, 28)),
            tf.keras.layers.Dense(128, activation="relu"),
         tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(62, activation="softmax")])
    model.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
    return model
    
    
'''
'''
def get_model(lr):
    # Create a custom optimizer with SGD and momentum
    #optimizer = tf.keras.optimizers.SGD(learning_rate=lr, decay = lr/100, momentum=0.9)
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
    # Build the model
    model = tf.keras.models.Sequential([
        tf.keras.layers.Input(input_shape=(28, 28, 1)),
        tf.keras.layers.Conv2D(5, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(62, activation="softmax")
    ])

    # Compile the model using the custom optimizer and categorical cross-entropy
    model.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"])

    return model


'''
def get_model(lr):
    # Create a custom optimizer with SGD and momentum
    #optimizer = tf.keras.optimizers.SGD(learning_rate=lr, decay = lr/100, momentum=0.9)
    optimizer = tf.keras.optimizers.SGD(learning_rate=lr)
    
    he_initializer = tf.keras.initializers.HeNormal()

    # Build the model with He initializer for the Dense layers
    model = tf.keras.models.Sequential([
        tf.keras.layers.Flatten(input_shape=(28, 28)),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dense(62, activation="softmax")
    ])

    # Compile the model using the custom optimizer and categorical cross-entropy
    model.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"])

    return model


## Splitting Data

In [26]:
all_train = []
all_val = []
x_train_all = []
y_train_all = []
x_val_all =[]
y_val_all =[]

for i in range(100):
    
    data = train_data_arr[i]
    # Split the data into training and validation sets
    images_train, images_val, labels_train, labels_val = train_test_split(
        data['images'],
        data['labels'],
        test_size=0.2,
        random_state=42)
   
    # Assuming you have a list of images (train_images, test_images)
    # Assuming images_train is a NumPy array
    '''
    images_train = np.array(images_train)
    images_val = np.array(images_val)

    # Add a new axis to represent the channel
    images_train = np.expand_dims(images_train, axis=-1).astype('float32')
    images_val = np.expand_dims(images_val, axis=-1).astype('float32')
    '''

    #train_dataset = tf.data.Dataset.from_tensor_slices((images_train, labels_train))
    #val_dataset = tf.data.Dataset.from_tensor_slices((images_val, labels_val))
    train_dataset = {'images': np.array(images_train),'labels':np.array(labels_train)}
    val_dataset =  {'images': np.array(images_val),'labels':np.array(labels_val)}
    all_train.append(train_dataset)
    all_val.append(val_dataset)


# Client class

In [27]:
###with shuffling
@ray.remote(num_cpus=1, num_gpus=1)
class Client:
    def __init__(self, data, batch_size, lr, E):
        # Create model
        self.model = get_model(lr)

        # Save the data references
        self.training_data_ref = ray.put(data[0])
        self.validation_data_ref = ray.put(data[1])
        self.batch_size = batch_size
        self.E = E

    def train(self, global_weights):
        # Retrieve the data from the references
        training_data = ray.get(self.training_data_ref)
        validation_data = ray.get(self.validation_data_ref)

        # Shuffle training data
        shuffled_train_indices = np.random.permutation(len(training_data['images']))
        shuffled_train_images = training_data['images'][shuffled_train_indices]
        shuffled_train_labels = training_data['labels'][shuffled_train_indices]

        # Set global weights
        self.model.set_weights(global_weights)
        
        # Use shuffled NumPy arrays for training
        history = self.model.fit(
            x=shuffled_train_images,
            y=shuffled_train_labels,
            batch_size=self.batch_size,
            epochs=self.E,
            validation_data=(validation_data['images'], validation_data['labels']),
            verbose=0
        )
        return self.model.get_weights()

    def evaluate(self):
        # Retrieve the data from the references
        training_data = ray.get(self.training_data_ref)
        validation_data = ray.get(self.validation_data_ref)

        # Shuffle validation data
        shuffled_val_indices = np.random.permutation(len(validation_data['images']))
        shuffled_val_images = validation_data['images'][shuffled_val_indices]
        shuffled_val_labels = validation_data['labels'][shuffled_val_indices]

        # Use shuffled NumPy arrays for evaluation
        train_loss, train_accuracy = self.model.evaluate(
            x=training_data['images'],
            y=training_data['labels'],
            batch_size=self.batch_size, 
            verbose=0
        )
        val_loss, val_accuracy = self.model.evaluate(
            x=shuffled_val_images,
            y=shuffled_val_labels,
            batch_size=self.batch_size, 
            verbose=0
        )
        return train_loss, train_accuracy, val_loss, val_accuracy


# Aggregated metrics and losses
Aggregate client metrics and losses for (federated) evaluation metrics.

In [28]:
sample_nbs = []
for i in range(100):
    sample_nbs.append(len(train_data_arr[i]['labels']))

In [29]:
#sample_nbs

In [30]:
def weighted_average(sampled_clients, client_results, sample_nbs):
    # TODO: Aggregate client losses or accuracies by taking a weighted average.
    #       Weights are proportional to the number of samples at each client
    sample_size_selected = [sample_nbs[c] for c in sampled_clients]
    #sorted list so order should match
    #train_loss, train_accuracy, val_loss, val_accuracy
    #train_loss, train_accuracy, val_loss, val_accuracy
    train_loss=[]
    train_accuracy=[]
    val_loss=[]
    val_accuracy=[]
    for i in range(0, len(client_results)):
        train_loss.append(client_results[i][0])
        train_accuracy.append(client_results[i][1])
        val_loss.append(client_results[i][2])
        val_accuracy.append(client_results[i][3])
        
    agg_train_acc = sum(weight * acc for weight, acc in zip(sample_size_selected,train_accuracy)) / sum(sample_size_selected)
    agg_train_loss = sum(weight * loss for weight, loss in zip(sample_size_selected,train_loss )) / sum(sample_size_selected)
    agg_val_acc = sum(weight * acc for weight, acc in zip(sample_size_selected, val_accuracy)) / sum(sample_size_selected)
    agg_val_loss = sum(weight * loss for weight, loss in zip(sample_size_selected, val_loss)) / sum(sample_size_selected)


    return agg_train_acc, agg_train_loss, agg_val_acc, agg_val_loss

# Aggregation of client updates

In [31]:
def fedAvg(sampled_clients, client_weights, sample_nbs):
    # Get the size of samples for each client of the selected clients
    sample_size_selected = [sample_nbs[c] for c in sampled_clients]
    #print("shape of local weights: ", len(client_weights[0]), client_weights[0][0].shape)
    list_dim = len(client_weights[0]) # get number or elements in the list.  
    #np_dim = client_weights[0][0].shape
    total = sum(sample_size_selected) # total sample size of all clients that are selected
    
    avg_weights = [] # this will be the updated avg weights 
    # for each numpy array in the list of weights 
    for k in range(list_dim): 
        #create a zeros numpy array of same shape. 
        avg_weights.append(np.zeros(client_weights[0][k].shape))
        #print(client_weights[0][k].shape)
    for i in range (len(sampled_clients)):
        #print("client: ", sampled_clients[i])
        #for each client, define scalar as its sample size/total sample size
        scalar = sample_size_selected[i]/total
        #print(scalar)
        #print(sample_size_selected[i])
        for j in range(0, list_dim):
            #for each numpy array in list
            #print("local np.array: ", client_weights[i][j])
            this_numpy = np.multiply(client_weights[i][j], scalar)
            #print("after multiplying by scalar:  ", this_numpy)
            avg_weights[j] = np.add(avg_weights[j],this_numpy)  
            #print("updated avg weights: ", avg_weights[j])
    #print("shape of avg weights: ", len(avg_weights), avg_weights[0].shape)
    return avg_weights


In [32]:
def plots_train(run, t, client, E,  train_loss, train_accuracy, val_loss, val_accuracy):
    plt.figure(figsize=(12, 4))
    
    plt.plot(train_loss, label='Training Loss')
    plt.plot(val_loss, label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Rounds')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig(run+ 'train_loss_'+str(t)+'_'+str(client)+'_'+str(E)+'_.png')
    plt.close()
    # Plot training accuracy and validation accuracy
    plt.figure(figsize=(12, 4))
    plt.plot(train_accuracy, label='Training Accuracy')
    plt.plot(val_accuracy, label='Validation Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Rounds')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.savefig(run+'train_accuracy_'+str(t)+'_'+str(client)+'_'+str(E)+'_.png')
    plt.close()

# Launch the learning

In [33]:
run = "p2_run3_new_"
Agg_metric = {}
lr = 0.0005 #0.00005
Agg_metric['train_loss'] = []
Agg_metric['train_accuracy'] =[]
Agg_metric['val_loss'] =[]
Agg_metric['val_accuracy'] =[]
print("passed")
    
# Initialize the global model weights
batch_size = 8
global_model = get_model(lr)
users = np.arange(100)
# Get data
#data_by_clients, test_data = partition_data()

total_rounds = 500
print("passed")
C = 4   # Only a fraction of C of clients participate in the local training per round
E = 40    # Number of epochs for each local training

passed
passed


2023-11-15 05:47:26.953190: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-11-15 05:47:28.963992: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1616] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 7377 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1080, pci bus id: 0000:05:00.0, compute capability: 6.1
2023-11-15 05:47:28.964584: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1616] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 7377 MB memory:  -> device: 1, name: NVIDIA GeForce GTX 1080, pci bus id: 0000:06:00.0, compute capability: 6.1
2023-11-15 05:47:28.965006: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1616] Created device /job:localhost/replica:0/task:0/de

In [34]:

round = 0 
Agg_metric['val_accuracy'].append(0)
#for round in range(total_rounds):
while round <total_rounds and Agg_metric['val_accuracy'][-1]<0.9:
    # Step 1
    if round ==0:
        # added a value just to pass the condition but need to initialize it back to an empty list
        Agg_metric['val_accuracy'] =[]

    
    #initiate empty list of model weights 
    client_weights =[]
    metric = {}
    metric['train_loss'] = []
    metric['train_accuracy'] =[]
    metric['val_loss'] =[]
    metric['val_accuracy'] =[] 
    # TODO: Sample C fraction of clients
    sampled_clients = np.random.choice(users, C, replace=False)
    sampled_clients = sorted(sampled_clients)
    sampled_data =[]
    for i in sampled_clients:
        sampled_data.append([all_train[i], all_val[i]])

    # TODO: Get the local data for the selected clients
    # Create Client instances for selected clients and assign local data
    clients = [Client.remote(data, batch_size, lr, E) for data in sampled_data]

    # Step 2: Perform local training and evaluaton on the clients
    client_weights = ray.get([client.train.remote(global_model.get_weights()) for client in clients])
    client_results = ray.get([client.evaluate.remote() for client in clients])
    
    '''
    metric_ = ray.put(metric)
    rewards = ray.get([get_reward.remote(action, metric_) for action in actions])
    '''
    
    #train_loss, train_accuracy, val_loss, val_accuracy
   
    # Step 3
    # TODO: Aggregate the losses and metrics, keep track of the metrics
    agg_train_acc, agg_train_loss, agg_val_acc, agg_val_loss = weighted_average(sampled_clients, client_results, sample_nbs)
    Agg_metric['train_loss'].append(agg_train_loss)
    Agg_metric['train_accuracy'].append(agg_train_acc)
    Agg_metric['val_loss'].append(agg_val_loss)
    Agg_metric['val_accuracy'].append(agg_val_acc)
    print ("round: ", round,  ",   aggregated validation acc: ", Agg_metric['val_accuracy'][round])
    # Step 4
    # TODO: Aggregate the client updates
    weighted_avg_weights = fedAvg(sampled_clients, client_weights, sample_nbs)
    
    # Step 5
    # Update the global model
    global_model.set_weights(weighted_avg_weights)
    
    #update round 
    round = round +1

ray.shutdown()

round:  0 ,   aggregated validation acc:  0.07398660760836222
round:  1 ,   aggregated validation acc:  0.09698293441990474
round:  2 ,   aggregated validation acc:  0.16941370121381485
round:  3 ,   aggregated validation acc:  0.13262749278339847
round:  4 ,   aggregated validation acc:  0.11469982797798178
round:  5 ,   aggregated validation acc:  0.1870767495029398
round:  6 ,   aggregated validation acc:  0.16667146702043528
round:  7 ,   aggregated validation acc:  0.24300737370537273
round:  8 ,   aggregated validation acc:  0.22341552016954136
round:  9 ,   aggregated validation acc:  0.2335584991969321
round:  10 ,   aggregated validation acc:  0.2335639383129495
round:  11 ,   aggregated validation acc:  0.2808041483875982
round:  12 ,   aggregated validation acc:  0.2413240688123331
round:  13 ,   aggregated validation acc:  0.28800973874607166
round:  14 ,   aggregated validation acc:  0.25610156416029867
round:  15 ,   aggregated validation acc:  0.3557791061404923
round:  

# Create plots

In [35]:
def test(test_data_arr, weights, lr):
    model = get_model(lr)  # Construct the model
    model.set_weights(weights)  # Set model weights with the latest parameters

    # Assuming test_data is a tuple of test_images and test_labels
    test_images = np.array(test_data_arr[0]['images'])
    test_labels = np.array(test_data_arr[0]['labels'])
    test_images_flat = test_images.reshape(test_images.shape[0], -1)

    test_dataset = tf.data.Dataset.from_tensor_slices((test_images, test_labels))
    batch_size = 16  # Set your desired batch size
    # Evaluate the model using the dataset
    batched_test_dataset = test_dataset.batch(batch_size)

    test_loss, test_accuracy = model.evaluate(batched_test_dataset)


    print(f"Test Accuracy: {test_accuracy:.4f}")

    return test_loss, test_accuracy

In [36]:
plots_train(run, total_rounds, C, E, Agg_metric['train_loss'] , Agg_metric['train_accuracy'], Agg_metric['val_loss'], Agg_metric['val_accuracy'])
# Evaluate on test data after training
test_loss, test_accuracy = test(test_data_arr, global_model.get_weights(), lr)

227/227 [==============================] - 1s 2ms/step - loss: 1.3918 - accuracy: 0.6443
Test Accuracy: 0.6443


In [37]:
print("test loss: ", test_loss)
print("test accuracy: ", test_accuracy)

test loss:  1.3917763233184814
test accuracy:  0.644297182559967


In [38]:
test_results = np.array ([test_loss, test_accuracy])
np.save('test_results_'+run+'_'+str(total_rounds)+'_'+str(C)+'_'+str(E)+'_'+str(batch_size)+'_.npy', test_results)